# X-RayDent — обучение Qwen3-0.6B с LoRA в Google Colab

Этот ноутбук клонирует проект, собирает датасет из 119 FAQ и safety-примеров, обучает LoRA-адаптер на GPU, проверяет веса и скачивает готовый ZIP.

Перед запуском выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**. Выполняйте ячейки сверху вниз. Токены и пароли вводить не требуется. После завершения отправьте файл `xraydent-qwen3-lora.zip`.

In [ ]:
# 1. Проверка GPU
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не найден. Включите T4 GPU в настройках среды выполнения Colab.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM, ГБ:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# 2. Установка библиотек. PyTorch из Colab не переустанавливаем.
# Предустановленный torchao 0.10 несовместим с PEFT 0.19 и для обычного LoRA не нужен.
%pip uninstall -y -q torchao
%pip install -q -U "transformers==4.57.6" "peft==0.19.1" "accelerate==1.14.0" "safetensors==0.8.0" "huggingface-hub==0.36.2"

## Настройки

Для первого качественного прогона оставьте значения по умолчанию. Датасет содержит точные и изменённые формулировки, а также отвлекающие FAQ-контексты. Три эпохи дают существенно больше optimizer steps, чем прежняя версия. Если Colab прервёт обучение, повторите ячейку обучения.

In [ ]:
# 3. Параметры обучения
REPO_URL = 'https://github.com/drSever/chat_bot_x-raydent.git'
BRANCH = 'main'
BASE_MODEL = 'Qwen/Qwen3-0.6B'
EPOCHS = 3
MAX_LENGTH = 640
SEED = 42

from pathlib import Path
PROJECT_DIR = Path('/content/chat_bot_x-raydent')
ADAPTER_DIR = PROJECT_DIR / 'artifacts' / 'adapter-colab'

In [ ]:
# 4. Получение актуального проекта и сборка датасета
import os
import shutil
import subprocess

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(['python', 'training/build_dataset.py'], check=True)

In [ ]:
# 5. Проверка датасета до обучения
import json

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

train_rows = read_jsonl('training/data/train.jsonl')
validation_rows = read_jsonl('training/data/validation.jsonl')
print('Обучающих примеров:', len(train_rows))
print('Проверочных примеров:', len(validation_rows))
print('Пример вопроса:', train_rows[0]['messages'][-2]['content'])
print('Пример ответа:', train_rows[0]['messages'][-1]['content'][:300])
assert len(train_rows) >= 300 and len(validation_rows) >= 30

In [ ]:
# 6. Обучение LoRA на полном датасете с полным выводом ошибок
import collections
import sys

command = [
    sys.executable, 'training/train_lora.py',
    '--profile', 'gpu',
    '--model', BASE_MODEL,
    '--epochs', str(EPOCHS),
    '--max-length', str(MAX_LENGTH),
    '--seed', str(SEED),
    '--output', str(ADAPTER_DIR),
]
print('Запуск:', ' '.join(command))
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1,
    env=environment,
)
last_lines = collections.deque(maxlen=80)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    last_lines.append(line.rstrip())
return_code = process.wait()
if return_code != 0:
    log_tail = '\n'.join(last_lines)
    lowered = log_tail.lower()
    if 'out of memory' in lowered:
        hint = 'Недостаточно видеопамяти. Уменьшите MAX_LENGTH до 384 и повторите ячейку.'
    elif 'cuda' in lowered and 'not available' in lowered:
        hint = 'Colab не предоставил GPU. Выберите T4 GPU и переподключите среду выполнения.'
    elif 'incompatible version of torchao' in lowered:
        hint = 'Удалите torchao: выполните `%pip uninstall -y torchao`, затем повторите обучение.'
    elif '401' in lowered or '403' in lowered or 'gated' in lowered:
        hint = 'Hugging Face отклонил загрузку модели. Проверьте доступ к BASE_MODEL.'
    else:
        hint = 'Скопируйте блок «Исходная ошибка Colab» и отправьте его разработчику.'
    raise RuntimeError(
        f'Обучение завершилось с кодом {return_code}. {hint}\n\n'
        f'--- Исходная ошибка Colab ---\n{log_tail}'
    )
torch.cuda.empty_cache()

In [ ]:
# 7. Проверка целостности полученных весов и метаданных
import hashlib
from safetensors import safe_open

weights_path = ADAPTER_DIR / 'adapter_model.safetensors'
config_path = ADAPTER_DIR / 'adapter_config.json'
metadata_path = ADAPTER_DIR / 'training_metadata.json'
assert weights_path.exists() and weights_path.stat().st_size > 1_000_000
assert config_path.exists() and metadata_path.exists()

with safe_open(str(weights_path), framework='pt', device='cpu') as weights:
    tensor_names = list(weights.keys())
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
sha256 = hashlib.sha256(weights_path.read_bytes()).hexdigest()
print(json.dumps(metadata, ensure_ascii=False, indent=2))
print('Тензоров:', len(tensor_names))
print('Размер весов, МБ:', round(weights_path.stat().st_size / 1024**2, 2))
print('SHA256:', sha256)
assert len(tensor_names) > 0

In [ ]:
# 8. Тест тем же grounded-промптом, который использует production API
from app.config import ROOT
from app.faq import parse_faq
from app.generator import SYSTEM_PROMPT, grounded_user_prompt
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
    trust_remote_code=False,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False)
model.eval()

entry = parse_faq(ROOT / 'data' / 'chatbot-faq-119.md')[0]
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': grounded_user_prompt(entry.question, [entry])},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print('Оригинальный FAQ:\n', entry.answer)
print('\nОтвет адаптера:\n', answer)
assert answer
original_words = set(entry.answer.lower().split())
answer_words = set(answer.lower().split())
word_overlap = len(original_words & answer_words) / max(1, len(original_words))
print('\nСовпадение слов с оригиналом:', round(word_overlap, 3))
assert word_overlap >= 0.55, 'Ответ слишком далеко ушёл от FAQ; увеличьте EPOCHS и повторите обучение.'

In [ ]:
# 9. Упаковка и скачивание результата
import gc
import zipfile
from google.colab import files

del model, base_model, tokenizer
gc.collect()
torch.cuda.empty_cache()

archive_path = Path('/content/xraydent-qwen3-lora.zip')
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ADAPTER_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, Path('adapter') / path.relative_to(ADAPTER_DIR))

print('Архив:', archive_path)
print('Размер, МБ:', round(archive_path.stat().st_size / 1024**2, 2))
print('После скачивания отправьте этот ZIP для внедрения в проект.')
files.download(str(archive_path))

## Что отправить после обучения

Отправьте файл **`xraydent-qwen3-lora.zip`** целиком. В архиве должны быть как минимум `adapter/adapter_model.safetensors`, `adapter/adapter_config.json` и `adapter/training_metadata.json`. Не переименовывайте и не редактируйте файлы внутри архива.